# Apartado 4. Análisis de subjetividad de comentarios

En este apartado, vamos a emplear un modelo preentrenado disponible en HuggingFace, concretamente [Twitter-RoBERTa-Base-Sentiment](https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment), para realizar un análisis de sentimientos sobre los comentarios extraídos de nuestros subreddits.

Este modelo está basado en la arquitectura RoBERTa y al que se le ha aplicado previamente un fine-tuning para tareas de clasificación de sentimientos sobre texto obtenido de la red social Twitter. Su objetivo es clasificar cada comentario en una de las tres categorías: Positivo, negativo, neutro.

El uso de este tipo de modelos preentrenados resulta especialmente útil en tareas de PLN, ya que nos permiten aprovechar conocimiento lingüístico aprendido previamente sobre grandes cantidades de texto, evitando tener que entrenar un modelo desde cero.
Además, al tratarse de embeddings contextuales, el modelo es capaz de interpretar el significado de una palabra dependiendo del contexto en el que aparece dentro de la frase, permitiendo capturar mejor matices, ironías o expresiones habituales en redes sociales.

Por simplicidad y con el fin de reducir el ruido presente en los comentarios, realizaremos el análisis sobre los datos preprocesados. Facilitando la tarea del modelo, permitiendo que se centre únicamente en el contenido semántico de cada comentario.

El sentimiento predicho para cada comentario, junto con las probabilidades asociadas a cada una de las clases, será almacenado posteriormente en los correspondientes ficheros JSON generados durante la práctica, llamados *sentimientos_limpio_[subreddit].json*.

In [1]:
!pip install transformers

In [2]:
from transformers import AutoModelForSequenceClassification
from transformers import AutoTokenizer
import numpy as np
from scipy.special import softmax
import csv
import urllib.request
import json
import torch

In [3]:
subreddits = ["limpio_books.json", "limpio_jobs.json", "limpio_LeagueOfLegends.json", "limpio_RandomThoughts.json",
			  "limpio_travel.json", "limpio_unpopularopinion.json"]

# Número de ejemplos que mostraremos por pantalla para comprobar manualmente los resultados obtenidos
MAX_EJEMPLOS = 5

# Tarea y modelo utilizados
task='sentiment'
MODEL = f"cardiffnlp/twitter-roberta-base-{task}"

# Cargamos el tokenizador del modelo
tokenizer = AutoTokenizer.from_pretrained(MODEL)

# Descargamos el mapeo de etiquetas del modelo
# Código obtenido del ejemplo presente en la plataforma Hugging Face
labels=[]
mapping_link = f"https://raw.githubusercontent.com/cardiffnlp/tweeteval/main/datasets/{task}/mapping.txt"
with urllib.request.urlopen(mapping_link) as f:
    html = f.read().decode('utf-8').split("\n")
    csvreader = csv.reader(html, delimiter='\t')
labels = [row[1] for row in csvreader if len(row) > 1]

# Cargamos el modelo para clasificación
model = AutoModelForSequenceClassification.from_pretrained(MODEL)

for subreddit in subreddits:
	with open(subreddit, "r", encoding="utf-8") as file:
		datos = json.load(file)

		print(f"\nEjemplos para el subreddit {datos["subreddit"]}:\n")

		sentimiento = {}
		ejemplos_mostrados = 0

		# Extraemos todos los hilos pertenecientes al subreddit
		hilos = datos["submissions"]

		for hilo in hilos:
			for comentario in hilo["comments"]:
				# Como el modelo tiene un límite máximo de 512 tokens, truncamos los comentarios que lo superan
				encoded_input = tokenizer(comentario["body"].lower(), return_tensors='pt', truncation=True, max_length=512)

				# Desactivamos el cálculo de gradientes ya que no estamos entrenando el modelo,
				# para reducir el consumo de memoria
				with torch.no_grad():
					output = model(**encoded_input)

		 			# Obtenemos los scores del modelo y aplicamos softmas para tradur en probabilidades
					scores = output[0][0].detach().numpy()
					scores = softmax(scores)

					# Ordenamos de mayor a menor
					ranking = np.argsort(scores)[::-1]

					# Añadimos el campo sentiment con el sentimiento más probable
					comentario["sentiment"] = labels[ranking[0]]

					# Almacenamos las probabilidades de cada sentimiento
					comentario["scores"] = {}

					for i in range(scores.shape[0]):
						l = labels[ranking[i]]
						s = scores[ranking[i]]
						comentario["scores"][l] = round(float(s), 4)

					# Mostramos algunos ejemplos por pantalla
					if ejemplos_mostrados < MAX_EJEMPLOS:
						print(f"\nComentario: {comentario["body"]}")
						print(f"Predicción: {comentario["sentiment"]}")
						print(f"Scores: {comentario["scores"]}")
						ejemplos_mostrados += 1

		# Guardamos el json modificado
		nombre_salida = f"sentimientos_{subreddit}"

		with open(nombre_salida, "w", encoding="utf-8") as f:
			json.dump(datos, f, ensure_ascii=False, indent=4)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/747 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Ejemplos para el subreddit books:


Comentario: You 're preaching choir
Predicción: neutral
Scores: {'neutral': 0.699, 'negative': 0.2283, 'positive': 0.0727}

Comentario: Does listening radio count reading Especially song like Devil Went Down Georgia tells story Does listening somebody talk count reading It silly conversation counts someone wants ask questions
Predicción: neutral
Scores: {'neutral': 0.8033, 'negative': 0.1411, 'positive': 0.0556}

Comentario: refer audiobooks bookbooks talking friends It 's stupid 's stuck also give anyone says `` audiobooks n't real books/reading '' quick boot arse say r/Discworld sub We class form ableism tbh Consume media choose consume Watch films subtitles listen books play computer games console choice As long 're happy 's important thing
Predicción: negative
Scores: {'negative': 0.5493, 'neutral': 0.3757, 'positive': 0.075}

Comentario: Are many people saying otherwise
Predicción: neutral
Scores: {'neutral': 0.6223, 'negative': 0.3531, 'positi